### Variables
####
Variables are unknown or changing parts of a model, the values taken by the variables are referred to as a solution and the output of an optimization process
### Parameters
####
The data that is needed to perform the optimization is referred to as data. 
### Relations
####
Equations, inequalities, or other mathematical relationships that define how different parts of a model connect to each other
### Goals
####
Functions that reflect goals and objectives for the system being modelled 
### Steps taken 
Create model and declare components -> instantiate the model -> apply the solver -> interrogate solver results
### Set
####
set data that is used to define a model instance 
### Param
####
parameter data that is used to define a model instance 
### Var
####
decision variables in a model 
### Constraint 
####
constraint epxressions that impose restrictions on variable values in a model  


## Abstract Versus Concrete models

A mathematical model can be defined using symbols that represent data values. For example, the following equations represent a linear program (LP) to find optimal values for the vector $x$ with parameters $n$ and $b$, and parameter vectors $a$ and $c$: 

$$
\begin{array}{rll}
\min & \sum_{j=1}^{n} c_j x_j & \\
\textrm{s.t.} & \sum_{j=1}^{n} a_{ij} x_j \geq b_i & \forall i = 1 \ldots m \\
& x_j \geq 0 & \forall j = 1 \ldots n
\end{array}
$$

In many contexts, a mathematical model can and should be directly defined with
the data values supplied at the time of the model definition. We call these
*concrete* mathematical models. For example, the following LP model is a
concrete instance of the previous abstract model:

$$
\begin{array}{rl}
\min & 2x_1 + 3x_2 \\
\textrm{s.\,t.} & 3x_1 + 4x_2 \geq 1 \\
& x_1, x_2 \geq 0
\end{array}
$$

The `ConcreteModel` class is used to define concrete optimization models in Pyomo.

In [2]:
# We test a simple concrete model 

import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.x = pyo.Var([1, 2], domain=pyo.NonNegativeReals)
model.OBJ = pyo.Objective(expr = 2*model.x[1] + 3*model.x[2])
model.Constraint1 = pyo.Constraint(expr = 3*model.x[1] + 4*model.x[2] >= 1)


In [ ]:
# We test a simple abstract pyomo model 

import pyomo.environ as pyo

# we declare an arbitrary abstract model, note that other names can be substituted in place of model
model = pyo.AbstractModel()
# Next, we declare parameters m and n using Param. within forces internal validation by Pyomo
model.m = pyo.Param(within=pyo.NonNegativeIntegers) # note that we have m number of constraints, indexed by the set i
model.n = pyo.Param(within=pyo.NonNegativeIntegers) # and n number of variables, indexed by the set j

# We define indexed sets that starts at 1 and ends ata value specified by the parameters model.m and model.n
model.I = pyo.RangeSet(1, model.m)
model.J = pyo.RangeSet(1, model.n)

# Next, we define a, b, c as indexed parameters. If we give sets to the Param component, an index is given to our parameter
# Declare coefficient matrix a mxn matrix
model.a = pyo.Param(model.I, model.J)
model.b = pyo.Param(model.I)
model.c = pyo.Param(model.J)

# We define a variable x, we also supply a set to the argument, therefore, we get variable that is indexed (by a set)
# Second argument defines the domain for variable, which is our constraint
# next, we define a variable indexed by the set J
model.x = pyo.Var(model.J, domain=pyo.NonNegativeReals)

# In abstract models, we define objective(s) and constraint(s) declarations via a function defined with a Python def statement 
# Recall first year knowledge, that the def statement establishes a name for a function along with its arguments
# And that when Pyomo uses a function to get objective or constraint expression, the model is always passsed as the first model, additional arguments if needed are supplied 

def obj_expression(m):
    # Returns 1x1
    return pyo.summation(m.c, m.x)

# Note that the default sense is minimization 
# unless sense=pyo.maximze argument is used
model.OBJ = pyo.Objective(rule=obj_expression)

# We declare the constraint expression similar to that of the objective expression
# Note that it is easy to generate multiple constraints that have the same type of expression 
# Note that in this case, it is important to state that we want to have a constraint for every i, parameterized from 1 to m
# The index i is explicitly included as a parameter for the constraint expression )
def ax_constraint_rule(m, i):
    # Returns 1x1
    # return the expression for the constraint for i
    return sum(m.a[i, j] * m.x[j] for j in m.J) >= m.b[i]

# the next line creates one constraint for each member of the set model.I
model.AxbConstraint = pyo.Constraint(model.I, rule=ax_constraint_rule)
